# Databricks Genie Space Migration Utility

**Purpose:** Clone an existing Genie Space into another AppSpace/catalog/schema in the **same Databricks workspace**, while retaining its serialized configuration and remapping fully qualified data identifiers.

This notebook is designed to be configuration-driven. Edit **Section 1** and run the notebook from top to bottom.

## What it does

1. Reads the source Genie Space configuration using the Genie Space API.
2. Saves a backup of the source serialized configuration.
3. Rewrites references from the source `catalog.schema` to the target `catalog.schema` throughout the serialized JSON, including data-source identifiers and SQL/instruction strings.
4. Checks that destination tables/views exist and are accessible from the notebook identity (optional but recommended).
5. Produces a dry-run report.
6. Creates a **new** Genie Space only when you explicitly enable creation and type the confirmation phrase.
7. Reads the created Genie back and verifies key configuration elements.

## Important limitations and safeguards

- The source Genie is never modified.
- The migration copies the serialized configuration returned by Databricks. It does not copy conversations, message history, permissions/ACLs, or external workspace objects.
- Only three-part identifiers matching the configured source catalog and schema are remapped. Unqualified names and references to other catalogs/schemas are intentionally left unchanged.
- SQL is transformed as text within the serialized JSON. Review SQL examples, joins, SQL functions and benchmark SQL before creating the target Genie, especially if your SQL uses unusual quoting, dynamic SQL, or split identifier fragments.
- A successful API create does **not** prove every query works. Test representative questions in the new Genie.
- The notebook's Spark identity must be able to describe destination objects if destination validation is enabled. Genie users also need their own required Unity Catalog permissions.
- API request fields and serialized format can evolve. This notebook uses the current Genie Space Management API documented by Databricks.

**API reference:** https://docs.databricks.com/api/genie/v1/genie-space

## 1. Configuration

Change values in this cell only for a normal migration.

**Authentication:** The notebook uses a Databricks secret scope. Create secrets for the workspace host and a token with Genie API access. The token must have permission to read the source Genie and create the destination Genie. Do not hard-code a token in this notebook.

For production, use your organization's approved OAuth/service-principal authentication approach where available.

In [ ]:
# =========================
# USER CONFIGURATION
# =========================

CONFIG = {
    # Databricks workspace
    "workspace_host": "https://<your-workspace-host>",

    # Authentication: set these to your existing secret scope and keys.
    "secret_scope": "my_secret_scope",
    "host_secret_key": "workspace_url",
    "token_secret_key": "api_token",

    # Source Genie Space in AppSpace ABC
    "source_genie_id": "<ABC_GENIE_SPACE_ID>",
    "source_catalog": "abc_catalog",
    "source_schema": "schema1",

    # Destination data in AppSpace XYZ
    "target_catalog": "xyz_catalog",
    "target_schema": "schema1",

    # New Genie properties
    "target_genie_title": "XYZ Analytics Genie",
    "target_genie_description": "Migrated Genie configuration for XYZ.",
    "target_warehouse_id": "<DESTINATION_SQL_WAREHOUSE_ID>",

    # Optional workspace folder for the new Genie.
    # Leave as None to use the default location.
    "target_parent_path": None,

    # Preflight checks
    "validate_destination_objects_with_spark": True,
    "fail_if_source_references_remain": True,

    # Safety switches:
    # Keep False for the first run. Review the report before enabling.
    "create_new_genie": False,
    "creation_confirmation": "",  # Must be exactly: CREATE XYZ GENIE

    # Optional output directory in the notebook driver's local filesystem.
    # Files are written here as downloadable notebook artifacts when supported.
    "output_dir": "/tmp/genie_migration"
}

# Basic validation of required configuration
required_values = [
    "workspace_host", "source_genie_id",
    "source_catalog", "source_schema",
    "target_catalog", "target_schema",
    "target_genie_title", "target_warehouse_id"
]
missing = [
    key for key in required_values
    if not CONFIG.get(key) or str(CONFIG.get(key)).startswith("<")
]
if missing:
    raise ValueError(
        "Please complete these CONFIG values before running: "
        + ", ".join(missing)
    )

if CONFIG["source_catalog"].lower() == CONFIG["target_catalog"].lower() and \
   CONFIG["source_schema"].lower() == CONFIG["target_schema"].lower():
    raise ValueError("Source and target catalog.schema are identical. Check CONFIG.")

print("Configuration validated.")
print(
    f"Mapping: {CONFIG['source_catalog']}.{CONFIG['source_schema']} "
    f"-> {CONFIG['target_catalog']}.{CONFIG['target_schema']}"
)

## 2. Imports, authentication and API helper

The `GET` request must include `include_serialized_space=true`. The source Genie requires at least **CAN EDIT** permission to export its serialized configuration.

In [ ]:
import copy
import json
import os
import re
import requests
from datetime import datetime, timezone
from pathlib import Path

try:
    # Databricks notebook environment
    dbutils  # noqa: F821
except NameError:
    raise RuntimeError(
        "Run this notebook in a Databricks notebook environment, "
        "or replace the secret retrieval block with your approved authentication method."
    )

secret_host = dbutils.secrets.get(
    scope=CONFIG["secret_scope"],
    key=CONFIG["host_secret_key"]
).rstrip("/")

token = dbutils.secrets.get(
    scope=CONFIG["secret_scope"],
    key=CONFIG["token_secret_key"]
)

# Secret host takes precedence, but it must match the configured workspace.
configured_host = CONFIG["workspace_host"].rstrip("/")
if configured_host.startswith("https://<"):
    raise ValueError("Set CONFIG['workspace_host'] to your Databricks workspace URL.")

if secret_host and secret_host != configured_host:
    raise ValueError(
        "The workspace URL in the secret does not match CONFIG['workspace_host']. "
        "Update one so they refer to the same workspace."
    )

WORKSPACE_HOST = configured_host
API_BASE = f"{WORKSPACE_HOST}/api/2.0/genie/spaces"

HEADERS = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

def genie_api(method, path="", *, params=None, payload=None, timeout=90):
    url = API_BASE + path
    response = requests.request(
        method=method.upper(),
        url=url,
        headers=HEADERS,
        params=params,
        json=payload,
        timeout=timeout
    )

    if not response.ok:
        try:
            detail = response.json()
        except Exception:
            detail = response.text
        raise RuntimeError(
            f"Genie API {method.upper()} {url} failed "
            f"(HTTP {response.status_code}): {detail}"
        )

    if response.status_code == 204 or not response.text.strip():
        return {}
    return response.json()

print("Authentication configured. Token is not displayed.")

## 3. Export source Genie configuration

This retrieves the full serialized configuration and keeps a local backup. The export is the master input for this migration run.

In [ ]:
def export_source_genie(source_genie_id):
    metadata = genie_api(
        "GET",
        f"/{source_genie_id}",
        params={"include_serialized_space": "true"}
    )

    serialized = metadata.get("serialized_space")
    if not serialized:
        raise ValueError(
            "The API response did not include serialized_space. "
            "Confirm source Genie ID and CAN EDIT permission."
        )

    try:
        source_config = json.loads(serialized)
    except json.JSONDecodeError as exc:
        raise ValueError(
            "serialized_space was returned but is not valid JSON."
        ) from exc

    if not isinstance(source_config, dict):
        raise ValueError("Expected serialized_space to decode to a JSON object.")

    if "version" not in source_config:
        raise ValueError(
            "The exported serialized configuration has no 'version' field. "
            "Review the API response before continuing."
        )

    return metadata, source_config, serialized


source_metadata, source_config, source_serialized = export_source_genie(
    CONFIG["source_genie_id"]
)

print("Source Genie:", source_metadata.get("title"))
print("Source Genie ID:", source_metadata.get("space_id"))
print("Serialized configuration version:", source_config.get("version"))
print("Top-level configuration keys:", list(source_config.keys()))

# Save original export for audit/recovery.
os.makedirs(CONFIG["output_dir"], exist_ok=True)
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
backup_path = os.path.join(
    CONFIG["output_dir"],
    f"source_genie_{CONFIG['source_genie_id']}_{run_stamp}.json"
)
with open(backup_path, "w", encoding="utf-8") as f:
    json.dump(source_config, f, ensure_ascii=False, indent=2)

print("Source configuration backup:", backup_path)

## 4. Inspect source data sources

Genie serialized configurations commonly contain `data_sources.tables` and `data_sources.metric_views`. This helper reports the configured identifiers and column configuration counts without assuming every optional section exists.

In [ ]:
def list_data_sources(serialized_config):
    data_sources = serialized_config.get("data_sources") or {}
    rows = []

    for source_type in ("tables", "metric_views"):
        for item in data_sources.get(source_type, []) or []:
            if not isinstance(item, dict):
                continue
            column_configs = item.get("column_configs") or []
            rows.append({
                "source_type": source_type,
                "identifier": item.get("identifier"),
                "column_config_count": len(column_configs),
                "has_description": bool(item.get("description"))
            })
    return rows

source_data_sources = list_data_sources(source_config)

if not source_data_sources:
    raise ValueError(
        "No tables or metric_views were found in the exported Genie configuration. "
        "Inspect the source export before proceeding."
    )

print(f"Found {len(source_data_sources)} configured data source(s):")
for row in source_data_sources:
    print(row)

## 5. Transform fully qualified identifiers

The replacement is restricted to three-part names of the form:

- `abc_catalog.schema1.table_name`
- `` `abc_catalog`.`schema1`.`table_name` ``

It preserves backticks around each identifier component where they were present. It changes only the configured source catalog and schema; table/function names remain unchanged.

The transformation walks the serialized JSON recursively, so references in data-source identifiers, SQL examples, instructions, joins, snippets and benchmark SQL are considered. It does not change plain column names or synonyms unless they themselves contain a matching fully qualified identifier.

In [ ]:
def compile_identifier_pattern(source_catalog, source_schema):
    # Match optional backticks independently for catalog, schema and object.
    # Backreferences ensure that a component is either quoted on both sides or not quoted.
    catalog = re.escape(source_catalog)
    schema = re.escape(source_schema)

    pattern = (
        r"(?<![A-Za-z0-9_$])"
        r"(?P<cq>`?)(?P<catalog>" + catalog + r")(?P=cq)"
        r"\s*\.\s*"
        r"(?P<sq>`?)(?P<schema>" + schema + r")(?P=sq)"
        r"\s*\.\s*"
        r"(?P<tq>`?)(?P<object>[A-Za-z_][A-Za-z0-9_$]*)(?P=tq)"
        r"(?![A-Za-z0-9_$])"
    )

    return re.compile(pattern, flags=re.IGNORECASE)


def replace_identifier_in_text(
    text,
    source_catalog,
    source_schema,
    target_catalog,
    target_schema
):
    pattern = compile_identifier_pattern(source_catalog, source_schema)

    def replacement(match):
        return (
            f"{match.group('cq')}{target_catalog}{match.group('cq')}."
            f"{match.group('sq')}{target_schema}{match.group('sq')}."
            f"{match.group('tq')}{match.group('object')}{match.group('tq')}"
        )

    return pattern.sub(replacement, text)


def transform_json_strings(
    value,
    source_catalog,
    source_schema,
    target_catalog,
    target_schema,
    path="$",
    changes=None
):
    if changes is None:
        changes = []

    if isinstance(value, dict):
        return {
            key: transform_json_strings(
                item,
                source_catalog, source_schema,
                target_catalog, target_schema,
                path=f"{path}.{key}",
                changes=changes
            )
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            transform_json_strings(
                item,
                source_catalog, source_schema,
                target_catalog, target_schema,
                path=f"{path}[{index}]",
                changes=changes
            )
            for index, item in enumerate(value)
        ]

    if isinstance(value, str):
        new_value = replace_identifier_in_text(
            value,
            source_catalog, source_schema,
            target_catalog, target_schema
        )
        if new_value != value:
            changes.append({
                "path": path,
                "before": value,
                "after": new_value
            })
        return new_value

    return value


transformation_log = []
target_config = transform_json_strings(
    copy.deepcopy(source_config),
    CONFIG["source_catalog"],
    CONFIG["source_schema"],
    CONFIG["target_catalog"],
    CONFIG["target_schema"],
    changes=transformation_log
)

print("Transformation complete.")
print("Serialized string values updated:", len(transformation_log))

for item in transformation_log[:20]:
    print(f"- {item['path']}: {item['before']} -> {item['after']}")

if len(transformation_log) > 20:
    print(f"... and {len(transformation_log) - 20} additional change(s).")

## 6. Check for source references that remain

This check searches all serialized JSON strings for a remaining fully qualified identifier using the configured source `catalog.schema`.

If the source pair appears in comments, descriptions or intentional cross-references, inspect those results. Do not blindly replace unrelated references.

In [ ]:
def find_remaining_source_references(
    value,
    source_catalog,
    source_schema,
    path="$",
    findings=None
):
    if findings is None:
        findings = []

    pattern = compile_identifier_pattern(source_catalog, source_schema)

    if isinstance(value, dict):
        for key, item in value.items():
            find_remaining_source_references(
                item, source_catalog, source_schema,
                path=f"{path}.{key}", findings=findings
            )
    elif isinstance(value, list):
        for index, item in enumerate(value):
            find_remaining_source_references(
                item, source_catalog, source_schema,
                path=f"{path}[{index}]", findings=findings
            )
    elif isinstance(value, str):
        for match in pattern.finditer(value):
            findings.append({
                "path": path,
                "reference": match.group(0)
            })

    return findings


remaining_source_refs = find_remaining_source_references(
    target_config,
    CONFIG["source_catalog"],
    CONFIG["source_schema"]
)

print("Remaining source catalog/schema references:", len(remaining_source_refs))
for item in remaining_source_refs:
    print(item)

if remaining_source_refs and CONFIG["fail_if_source_references_remain"]:
    raise RuntimeError(
        "Source references remain in the transformed configuration. "
        "Review the findings before continuing."
    )

## 7. Compare source and target data-source identifiers

The number of data sources should remain the same. Each configured source identifier should now point to the destination catalog and schema.

In [ ]:
target_data_sources = list_data_sources(target_config)

if len(source_data_sources) != len(target_data_sources):
    raise AssertionError(
        "Data-source count changed during transformation."
    )

print("SOURCE -> TARGET")
for source_row, target_row in zip(source_data_sources, target_data_sources):
    print(
        f"{source_row['identifier']} -> {target_row['identifier']} "
        f"(columns configured: {source_row['column_config_count']} "
        f"-> {target_row['column_config_count']})"
    )

# Confirm all source identifiers that begin with the configured source namespace
# have the expected destination namespace after transformation.
source_prefix = (
    CONFIG["source_catalog"] + "." + CONFIG["source_schema"] + "."
)
target_prefix = (
    CONFIG["target_catalog"] + "." + CONFIG["target_schema"] + "."
)

for row in target_data_sources:
    identifier = row.get("identifier")
    if not identifier:
        raise AssertionError("A data source is missing its identifier.")
    if identifier.lower().startswith(source_prefix.lower()):
        raise AssertionError(
            f"Unmapped data source identifier remains: {identifier}"
        )
    if identifier.lower().startswith(target_prefix.lower()):
        continue
    # Other catalog/schema references may be intentional; report them for review.
    print("REVIEW - identifier outside configured target namespace:", identifier)

## 8. Verify column-level configuration was preserved

This compares each source data source's `column_configs` with its transformed counterpart. It also compares descriptions. The only expected change to each data-source object is its fully qualified identifier and any other matching references embedded in its strings.

In [ ]:
def source_map_by_type_and_identifier(config):
    result = {}
    data_sources = config.get("data_sources") or {}

    for source_type in ("tables", "metric_views"):
        for item in data_sources.get(source_type, []) or []:
            identifier = item.get("identifier")
            if identifier:
                result[(source_type, identifier)] = item
    return result


source_map = source_map_by_type_and_identifier(source_config)
target_map = source_map_by_type_and_identifier(target_config)

column_checks = []
for (source_type, source_identifier), source_item in source_map.items():
    expected_target_identifier = replace_identifier_in_text(
        source_identifier,
        CONFIG["source_catalog"], CONFIG["source_schema"],
        CONFIG["target_catalog"], CONFIG["target_schema"]
    )
    target_item = target_map.get((source_type, expected_target_identifier))

    if target_item is None:
        column_checks.append({
            "source": source_identifier,
            "target": expected_target_identifier,
            "found": False,
            "column_configs_preserved": False,
            "description_preserved": False
        })
        continue

    column_checks.append({
        "source": source_identifier,
        "target": expected_target_identifier,
        "found": True,
        "column_configs_preserved":
            source_item.get("column_configs", []) ==
            target_item.get("column_configs", []),
        "description_preserved":
            source_item.get("description") ==
            target_item.get("description")
    })

for check in column_checks:
    print(check)

if not all(
    item["found"]
    and item["column_configs_preserved"]
    and item["description_preserved"]
    for item in column_checks
):
    raise AssertionError(
        "One or more data-source column configurations/descriptions "
        "were not preserved. Stop and inspect before creating the Genie."
    )

print("Column configuration and data-source descriptions preserved.")

## 9. Optional destination object validation using Spark

This runs `DESCRIBE TABLE` for each configured table/view identifier. It checks that the destination object exists and is visible to the notebook's current Spark identity.

If the Genie contains metric views, confirm that your Databricks Runtime and permissions support describing them in this way. Any failure must be investigated before creation.

In [ ]:
def quote_three_part_identifier(identifier):
    parts = identifier.split(".")
    if len(parts) != 3:
        raise ValueError(
            f"Expected a three-part Unity Catalog identifier, got: {identifier}"
        )
    # Remove surrounding backticks if present, then quote each part safely.
    cleaned = [part.strip().strip("`") for part in parts]
    return ".".join("`" + part.replace("`", "``") + "`" for part in cleaned)


destination_object_checks = []

if CONFIG["validate_destination_objects_with_spark"]:
    for row in target_data_sources:
        identifier = row["identifier"]
        try:
            quoted = quote_three_part_identifier(identifier)
            spark.sql(f"DESCRIBE TABLE {quoted}").limit(1).collect()
            destination_object_checks.append({
                "identifier": identifier,
                "status": "PASS",
                "error": None
            })
        except Exception as exc:
            destination_object_checks.append({
                "identifier": identifier,
                "status": "FAIL",
                "error": str(exc)[:1000]
            })

    for check in destination_object_checks:
        print(check)

    failed = [x for x in destination_object_checks if x["status"] != "PASS"]
    if failed:
        raise RuntimeError(
            "Destination object validation failed for one or more sources. "
            "Fix permissions/names or disable this check only if you have "
            "validated the objects another way."
        )
else:
    print("Destination object validation skipped by configuration.")

## 10. Save the transformed configuration and migration preview

This creates a target JSON file and a human-readable migration report before any Genie is created.

In [ ]:
target_backup_path = os.path.join(
    CONFIG["output_dir"],
    f"target_genie_config_{CONFIG['target_catalog']}_{CONFIG['target_schema']}_{run_stamp}.json"
)
with open(target_backup_path, "w", encoding="utf-8") as f:
    json.dump(target_config, f, ensure_ascii=False, indent=2)

migration_preview = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "source_genie_id": CONFIG["source_genie_id"],
    "source_genie_title": source_metadata.get("title"),
    "target_genie_title": CONFIG["target_genie_title"],
    "source_catalog": CONFIG["source_catalog"],
    "source_schema": CONFIG["source_schema"],
    "target_catalog": CONFIG["target_catalog"],
    "target_schema": CONFIG["target_schema"],
    "source_data_source_count": len(source_data_sources),
    "target_data_source_count": len(target_data_sources),
    "source_column_config_count": sum(
        row["column_config_count"] for row in source_data_sources
    ),
    "target_column_config_count": sum(
        row["column_config_count"] for row in target_data_sources
    ),
    "serialized_string_updates": len(transformation_log),
    "remaining_source_references": len(remaining_source_refs),
    "destination_object_validation": destination_object_checks,
    "target_config_backup": target_backup_path,
    "creation_enabled": CONFIG["create_new_genie"]
}

preview_path = os.path.join(
    CONFIG["output_dir"],
    f"migration_preview_{run_stamp}.json"
)
with open(preview_path, "w", encoding="utf-8") as f:
    json.dump(migration_preview, f, ensure_ascii=False, indent=2)

print(json.dumps(migration_preview, indent=2))
print("\nTarget configuration saved to:", target_backup_path)
print("Preview report saved to:", preview_path)

## 11. Create the new Genie Space (explicit opt-in)

**This is the only cell that creates a Genie Space.**

To create:
1. Review all previous output.
2. Set `CONFIG["create_new_genie"] = True` in Section 1.
3. Set `CONFIG["creation_confirmation"] = "CREATE XYZ GENIE"` in Section 1.
4. Re-run the notebook from the beginning so the export and validation are fresh.

The source Genie is not updated. The API creates a separate Genie Space from the transformed serialized configuration.

In [ ]:
def create_target_genie(target_config, config):
    serialized_target = json.dumps(
        target_config,
        ensure_ascii=False,
        separators=(",", ":")
    )

    payload = {
        "title": config["target_genie_title"],
        "description": config["target_genie_description"],
        "warehouse_id": config["target_warehouse_id"],
        "serialized_space": serialized_target
    }

    if config.get("target_parent_path"):
        payload["parent_path"] = config["target_parent_path"]

    return genie_api(
        "POST",
        "",
        payload=payload,
        timeout=180
    )


if not CONFIG["create_new_genie"]:
    print(
        "CREATE SKIPPED. Set create_new_genie=True and "
        "creation_confirmation='CREATE XYZ GENIE' in Section 1 "
        "after reviewing the migration preview."
    )
elif CONFIG["creation_confirmation"] != "CREATE XYZ GENIE":
    raise RuntimeError(
        "Creation is enabled, but the confirmation phrase is missing or incorrect."
    )
else:
    create_response = create_target_genie(target_config, CONFIG)
    new_genie_id = create_response.get("space_id")

    if not new_genie_id:
        raise RuntimeError(
            "Create API returned no space_id. Inspect the API response."
        )

    print("New Genie created.")
    print("Title:", create_response.get("title"))
    print("New Genie Space ID:", new_genie_id)
    print("Warehouse ID:", create_response.get("warehouse_id"))

    # Save the new Genie ID for later reference.
    id_path = os.path.join(
        CONFIG["output_dir"],
        f"created_genie_{run_stamp}.json"
    )
    with open(id_path, "w", encoding="utf-8") as f:
        json.dump(create_response, f, ensure_ascii=False, indent=2)

    print("Creation response saved to:", id_path)

## 12. Read back and verify the created Genie

Run this after successful creation. It compares the newly stored configuration with the intended target configuration for data-source identifiers and column configurations.

In [ ]:
def verify_created_genie(new_genie_id, expected_config):
    actual_metadata = genie_api(
        "GET",
        f"/{new_genie_id}",
        params={"include_serialized_space": "true"}
    )

    actual_serialized = actual_metadata.get("serialized_space")
    if not actual_serialized:
        raise ValueError(
            "Read-back response did not include serialized_space. "
            "Check permissions and API response."
        )

    actual_config = json.loads(actual_serialized)

    expected_rows = list_data_sources(expected_config)
    actual_rows = list_data_sources(actual_config)

    expected_ids = sorted(
        (x["source_type"], x["identifier"])
        for x in expected_rows
    )
    actual_ids = sorted(
        (x["source_type"], x["identifier"])
        for x in actual_rows
    )

    source_map_expected = source_map_by_type_and_identifier(expected_config)
    source_map_actual = source_map_by_type_and_identifier(actual_config)

    column_results = []
    for key, expected_item in source_map_expected.items():
        actual_item = source_map_actual.get(key)
        column_results.append({
            "source": key,
            "found": actual_item is not None,
            "column_configs_match": (
                actual_item is not None and
                expected_item.get("column_configs", []) ==
                actual_item.get("column_configs", [])
            ),
            "description_match": (
                actual_item is not None and
                expected_item.get("description") ==
                actual_item.get("description")
            )
        })

    verification = {
        "new_genie_id": new_genie_id,
        "title": actual_metadata.get("title"),
        "data_source_identifiers_match": expected_ids == actual_ids,
        "expected_data_source_count": len(expected_rows),
        "actual_data_source_count": len(actual_rows),
        "column_checks": column_results,
        "all_column_configs_match": all(
            x["found"] and x["column_configs_match"]
            for x in column_results
        ),
        "all_descriptions_match": all(
            x["found"] and x["description_match"]
            for x in column_results
        )
    }

    return verification, actual_config


if "new_genie_id" in globals():
    verification_report, readback_config = verify_created_genie(
        new_genie_id,
        target_config
    )
    print(json.dumps(verification_report, indent=2))

    if not (
        verification_report["data_source_identifiers_match"]
        and verification_report["all_column_configs_match"]
        and verification_report["all_descriptions_match"]
    ):
        print("WARNING: Read-back verification found differences. Review the report.")
else:
    print("No new_genie_id in this run. Create the Genie first, then run this section.")

## 13. Post-migration checklist

After creation, open the new Genie in the Databricks UI and complete these checks:

- [ ] New Genie appears in the intended workspace folder/AppSpace location.
- [ ] All expected XYZ tables and views appear as data sources.
- [ ] Column descriptions, synonyms and other column-level settings are present.
- [ ] General instructions and sample questions are present.
- [ ] Example SQL queries use the XYZ catalog/schema.
- [ ] Join specifications and SQL functions are present and refer to valid objects.
- [ ] Benchmark SQL (if configured) has been reviewed.
- [ ] Run several representative questions and inspect generated SQL/results.
- [ ] Confirm intended users have access to the Genie, SQL warehouse and XYZ Unity Catalog objects.

## 14. Reuse this notebook for another Genie

For a second Genie in the same target catalog/schema:

1. Keep the same source Genie ID and source catalog/schema.
2. Change `target_genie_title`, `target_genie_description`, and warehouse/folder if required.
3. Keep the same target catalog/schema if the second Genie uses the same XYZ objects.
4. Run the notebook again with creation enabled and a fresh confirmation.

**Recommended master-template practice:** Keep one reviewed master Genie (or its exported JSON in version control) as the canonical configuration. For each new target Genie, start from a fresh export of that master. Do not use a previously migrated Genie as the source unless that is intentional.

For a different target catalog/schema, only change `target_catalog` and `target_schema` in Section 1.

## API documentation

- [Genie Space Management API](https://docs.databricks.com/api/genie/v1/genie-space)
- [Databricks Genie Agents API guide](https://docs.databricks.com/aws/en/genie-agents/conversation-api)

---
**End of notebook.**